## Inference_Predict_Random

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import geopandas as gpd
import time
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.patches as mpatches

import random
from torch.utils.data import * 

In [2]:
if (torch.cuda.is_available()):
    device=torch.device("cuda")
    print('device is:',device)
else:
    device=torch.device("cpu")
    print('device is:',device)

device is: cpu


## Define Pytorch model class - Alpha_FC

In [3]:
# two fully connected layers and a softmax - 64 input features
class Alpha_FC(nn.Module):
    def __init__(self):                                               # instantiate in the constructor
        super().__init__()                                            # in - m x 10
        self.fc1 = nn.Linear(64, 32)                                  # fc1. out - m x 10
        self.act1 = nn.LeakyReLU()                                    # activation 1
        self.fc2 = nn.Linear(32, 16)                                  # fc2. out - m x 10
        self.act2 = nn.LeakyReLU()                                    # activation 2
        self.fc3 = nn.Linear(16,7)                                    # out - m x 7 - 7 classes

    def forward(self, x):                             
        out = self.act1(self.fc1(x))
        out = self.act2(self.fc2(out))
        out = self.fc3(out)       
        return out

## Define Pytorch model class - Patch

In [ ]:
# two conv layers and a fully connected layer and a softmax - Song plus FC layer
class Patch(nn.Module):
    def __init__(self):                                               # instantiate in the constructor
        super().__init__()                                            # in - m x 10 x 5 x 5
        self.conv1 = nn.Conv2d(10, 10, kernel_size=3, padding=0)      # out - m x 10 x 3 x 3
        self.act1 = nn.LeakyReLU()
        self.conv2 = nn.Conv2d(10, 20, kernel_size=2, padding=0)      # out - m x 20 x 2 x 2
        self.act2 = nn.LeakyReLU()
        # flatten
        self.fc1 = nn.Linear(80, 40)                                  # out - m x 40
        self.act3 = nn.LeakyReLU()
        self.fc2 = nn.Linear(40, 7)                                   # out logits - m x 7
                                                                      # common practice to output logits without activation function

    def forward(self, x):                            # uses the instances in forward
        out = self.act1(self.conv1(x))
        out = self.act2(self.conv2(out))
        out = out.view(-1, 80)                       # replacing nn.Flatten(), the size -1 is inferred from other dimensions
        out = self.act3(self.fc1(out))
        out = self.fc2(out)
        return out

## Instantiate a model and load saved weights from file

In [20]:
year = '2025'
#region_name = 'biomeF'
#region_name = 'biomeAll_southern_africa'
#region_name = 'biomeAll_eastern_sahel'
region_name = 'biomeAll_western_usa'
model_name = 'All'           

In [21]:
# AlphaEarth Classifier file name
file_name = '../Pixel_models/Alpha_biome' + model_name + '.pt'       # model is named after the biome
print(file_name)

../Pixel_models/Alpha_biomeAll.pt


In [22]:
# Load Alpha Model
loaded_model = Alpha_FC() 
loaded_model.load_state_dict(torch.load(file_name, weights_only=True))

<All keys matched successfully>

In [ ]:
# Patch CNN file name
file_name = '../Patch_models/Patch_biome' + model_name + '.pt'       # model is named after the biome
print(file_name)

In [ ]:
# Load Patch Model
loaded_model = Patch() 
loaded_model.load_state_dict(torch.load(file_name, weights_only=True))

In [23]:
model=loaded_model.to(device) 

## Import AlphaEarth random embeddings for biome and region

In [24]:
# filename
filepath = '../Inference_Points/' + region_name + '_random_embeddings_' + year + '.npy'       # biome B
filepath

'../Inference_Points/biomeAll_western_usa_random_embeddings_2025.npy'

In [25]:
# load random data
random_loaded = np.load(filepath)
random_loaded.shape

(37789, 64)

## Import random patch data for biome and region

In [ ]:
# filename
filepath = '../Inference_Patches/' + region_name + '_random_patches_' + year + '.npy'       # biome B
filepath

In [ ]:
# load random data
random_loaded_raw = np.load(filepath)
random_loaded_raw.shape

In [ ]:
random_loaded = random_loaded_raw/10000

random_loaded = np.minimum(random_loaded, 1.0)          # clip values above +1 stds
random_loaded = np.maximum(random_loaded, 0.0)          # clip values below 0 stds

## Run Inference on random points/patches

In [26]:
# run inference on image cube
start = time.time()

model.eval()

batch_t = torch.tensor(random_loaded, dtype=torch.float32)   # numpy to pytorch tensor
with torch.no_grad():
    batch_t = batch_t.to(device)    
    outputs = model(batch_t)
    pred_index = torch.argmax(outputs, dim=1)       

end = time.time()
print('final run time=',(end-start)/60," minutes")

final run time= 0.0005451440811157227  minutes


In [27]:
outputs.shape, pred_index.shape

(torch.Size([37789, 7]), torch.Size([37789]))

In [28]:
# pytorch tensor to numpy array
pred_index = pred_index.numpy()

In [29]:
count=np.zeros(7, dtype=int)

for i in range(7):
    count[i] = np.count_nonzero(pred_index == i)

print('year =', year)
print('count =', count)
print('total = ', np.sum(count))

year = 2025
count = [  430   684  2587  2751 11453 18834  1050]
total =  37789


## Save Count Data

In [30]:
# Alpha filename
filename = '../Inference_Points/' + region_name + '_model' + model_name + '_' + year + '_count.npy'
filename

'../Inference_Points/biomeAll_western_usa_modelAll_2025_count.npy'

In [ ]:
# CNN filename
filename = '../Inference_Patches/' + region_name + '_model' + model_name + '_' + year + '_patches_count.npy'
filename

In [31]:
# save count data
np.save(filename, count)

In [32]:
# load count data
count_loaded = np.load(filename)
count_loaded

array([  430,   684,  2587,  2751, 11453, 18834,  1050])

## Save Predicted Index Data

In [33]:
# Points filename
filename2 = '../Inference_Points/' + region_name + '_model' + model_name + '_pred_index_' + year + '.npy'
filename2

'../Inference_Points/biomeAll_western_usa_modelAll_pred_index_2025.npy'

In [ ]:
# Patches ilename
filename2 = '../Inference_Patches/' + region_name + '_model' + model_name + '_pred_index_' + year + '_patches.npy'
filename2

In [34]:
# save predicted index data
np.save(filename2, pred_index)

In [35]:
# load predicted index data
pred_index_loaded = np.load(filename2)
pred_index_loaded.shape

(37789,)

## Change Index

In [36]:
year1 = '2018'
year2 = '2025'
#region_name = 'biomeF'
#region_name = 'biomeAll_southern_africa'
#region_name = 'biomeAll_eastern_sahel'
region_name = 'biomeAll_western_usa'
model_name = 'All'  

In [37]:
# filename1
filename1 = '../Inference_Points/' + region_name + '_model' + model_name + '_pred_index_' + year1 + '.npy'
print(filename1)

# filename2
filename2 = '../Inference_Points/' + region_name + '_model' + model_name + '_pred_index_' + year2 + '.npy'
print(filename2)

../Inference_Points/biomeAll_western_usa_modelAll_pred_index_2018.npy
../Inference_Points/biomeAll_western_usa_modelAll_pred_index_2025.npy


In [38]:
# load predicted index data
pred_index1 = np.load(filename1)
print(pred_index1.shape)

pred_index2 = np.load(filename2)
print(pred_index2.shape)

(37789,)
(37789,)


### Create 7x7 Change Matrix

In [39]:
change_matrix = np.zeros((7, 7), dtype=int)

In [40]:
pred_index1.shape, pred_index2.shape

((37789,), (37789,))

In [41]:
for i in range(pred_index1.shape[0]):       
    change_matrix[pred_index1[i],pred_index2[i]] += 1


In [42]:
change_matrix

array([[  390,     0,     8,     3,     2,    31,     3],
       [    0,   552,    11,    10,    21,    47,     2],
       [    6,     6,  1494,     1,   116,    72,    11],
       [    2,    14,     6,  2331,   298,   145,    15],
       [    1,    20,   805,   216,  9991,  1056,     9],
       [   29,    84,   255,   175,  1016, 17199,   658],
       [    2,     8,     8,    15,     9,   284,   352]])

In [43]:
#np.save('../Inference_Change/biomeAll_southern_africa_modelAll_change_matrix.npy', change_matrix)
#np.save('../Inference_Change/biomeAll_eastern_sahel_modelAll_change_matrix.npy', change_matrix)
np.save('../Inference_Change/biomeAll_western_usa_modelAll_change_matrix.npy', change_matrix)

In [46]:
#loaded = np.load('../Inference_Change/biomeAll_southern_africa_modelAll_change_matrix.npy')
#loaded = np.load('../Inference_Change/biomeAll_eastern_sahel_modelAll_change_matrix.npy')
loaded = np.load('../Inference_Change/biomeAll_western_usa_modelAll_change_matrix.npy')

In [47]:
loaded

array([[  390,     0,     8,     3,     2,    31,     3],
       [    0,   552,    11,    10,    21,    47,     2],
       [    6,     6,  1494,     1,   116,    72,    11],
       [    2,    14,     6,  2331,   298,   145,    15],
       [    1,    20,   805,   216,  9991,  1056,     9],
       [   29,    84,   255,   175,  1016, 17199,   658],
       [    2,     8,     8,    15,     9,   284,   352]])